## Предобработка данных

### Загрузка данных

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import os


data_paths = [
    Path('../data/energydata_complete.csv'),
    Path('data/energydata_complete.csv'),
]
data_path = next((path for path in data_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError('Не найден data/energydata_complete.csv')

df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'], errors='raise')
df[['date', 'Appliances']].head()

,date,Appliances
0,2016-01-11 17:00:00,60
1,2016-01-11 17:10:00,60
2,2016-01-11 17:20:00,50
3,2016-01-11 17:30:00,50
4,2016-01-11 17:40:00,60


### Удаление шума и слабоинформативных метеопризнаков

In [13]:
noise_cols = ['rv1', 'rv2', 'Press_mm_hg', 'Windspeed', 'Visibility', 'Tdewpoint']
df_clean = df.drop(columns=noise_cols).copy()

print(f" Данные успешно загружены из {data_path}. Шум удален.")

 Данные успешно загружены из ../data/energydata_complete.csv. Шум удален.


### Временные признаки 

- **`weekday`** принимает значения от 0 (понедельник) до 6 (воскресенье). 
- **`is_weekend`** равен 1 для субботы и воскресенья. 
- **Пара `sin_hour` и `cos_hour`** помогает модели понимать, что 23:00 и 00:00 находятся рядом в суточном цикле.

In [3]:
df_clean['hour'] = df_clean['date'].dt.hour
df_clean['weekday'] = df_clean['date'].dt.weekday
df_clean['is_weekend'] = (df_clean['weekday'] >= 5).astype('int8')

df_clean['sin_hour'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['cos_hour'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

temporal_features = ['hour', 'weekday', 'is_weekend', 'sin_hour', 'cos_hour']
df_clean[['date', *temporal_features]].head(10)

,date,hour,weekday,is_weekend,sin_hour,cos_hour
0,2016-01-11 17:00:00,17,0,0,-0.965926,-2.588190e-01
1,2016-01-11 17:10:00,17,0,0,-0.965926,-2.588190e-01
2,2016-01-11 17:20:00,17,0,0,-0.965926,-2.588190e-01
3,2016-01-11 17:30:00,17,0,0,-0.965926,-2.588190e-01
4,2016-01-11 17:40:00,17,0,0,-0.965926,-2.588190e-01
5,2016-01-11 17:50:00,17,0,0,-0.965926,-2.588190e-01
6,2016-01-11 18:00:00,18,0,0,-1.000000,-1.836970e-16
7,2016-01-11 18:10:00,18,0,0,-1.000000,-1.836970e-16
8,2016-01-11 18:20:00,18,0,0,-1.000000,-1.836970e-16
9,2016-01-11 18:30:00,18,0,0,-1.000000,-1.836970e-16


### Признаки событий( душ, готовка) 
- Резкие кратковременные скачки влажности и температуры помогают фиксировать бытовые события: душ в ванной (`RH_5`) и готовку на кухне (`T1`, `RH_1`).
- **Дельты (`diff(1)`, `diff(2)`):** фиксируют направление и величину изменений за 10 и 20 минут по ванной (`RH_5`) и кухне (`T1`, `RH_1`)— положительные значения отражают рост показателей, отрицательные — спад после окончания активности или проветривания.
- **Положительный рост (`*_rise_*`):** отделяет резкий подъём от спада, чтобы модели было проще выделять начало бытового события.
- **Флаги сильных скачков (`is_*_spike_20m`):** отмечают верхние 2% по росту за 20 минут и позволяют проверить, сопровождаются ли такие всплески повышенным потреблением `Appliances` по сравнению с обычным фоном.
- **Проверка на модели:** окончательно оставляем в финальном наборе признаков те дельты/флаги, которые дадут прирост качества на временной валидации.

In [14]:
# Базовые изменения за 10 и 20 минут 
df_clean['shower_diff_10m'] = df_clean['RH_5'].diff(1)
df_clean['shower_diff_20m'] = df_clean['RH_5'].diff(2)

df_clean['cooking_temp_diff_10m'] = df_clean['T1'].diff(1)
df_clean['cooking_hum_diff_10m'] = df_clean['RH_1'].diff(1)
df_clean['cooking_hum_diff_20m'] = df_clean['RH_1'].diff(2)

# Только положительный рост (начало всплеска влажности/температуры)
df_clean['shower_rise_10m'] = df_clean['shower_diff_10m'].clip(lower=0)
df_clean['shower_rise_20m'] = df_clean['shower_diff_20m'].clip(lower=0)
df_clean['cooking_hum_rise_20m'] = df_clean['cooking_hum_diff_20m'].clip(lower=0)

# Флаги сильных скачков (верхние 2% по росту)
q_shower_20 = df_clean['shower_rise_20m'].quantile(0.98)
q_cook_20 = df_clean['cooking_hum_rise_20m'].quantile(0.98)

df_clean['is_shower_spike_20m'] = (df_clean['shower_rise_20m'] > q_shower_20).astype('int8')
df_clean['is_cooking_spike_20m'] = (df_clean['cooking_hum_rise_20m'] > q_cook_20).astype('int8')

# Смотрим, выше ли среднее/медианное потребление в моменты таких скачков
print("Медиана Appliances при скачке влажности в ванной (20м):",
      df_clean.loc[df_clean['is_shower_spike_20m'] == 1, 'Appliances'].median(),
      "vs обычный фон:",
      df_clean.loc[df_clean['is_shower_spike_20m'] == 0, 'Appliances'].median())

print("Медиана Appliances при скачке влажности на кухне (20м):",
      df_clean.loc[df_clean['is_cooking_spike_20m'] == 1, 'Appliances'].median(),
      "vs обычный фон:",
      df_clean.loc[df_clean['is_cooking_spike_20m'] == 0, 'Appliances'].median())

Медиана Appliances при скачке влажности в ванной (20м): 110.0 vs обычный фон: 60.0
Медиана Appliances при скачке влажности на кухне (20м): 120.0 vs обычный фон: 60.0


### Агрегатные признаки
- Используются внутренние датчики `T1`-`T5`, `T7`-`T9` и соответствующие им датчики влажности. `T6` и `RH_6` не включаются: по описанию датасета это внешние датчики на северной стороне здания.
-  Для температуры и влажности рассчитываются среднее, минимум, максимум и разброс между самым большим и самым маленьким значением. Также добавляются разницы между средней температурой/влажностью внутри дома и на улице.

In [5]:
indoor_sensor_numbers = [1, 2, 3, 4, 5, 7, 8, 9]
temperature_columns = [f'T{i}' for i in indoor_sensor_numbers]
humidity_columns = [f'RH_{i}' for i in indoor_sensor_numbers]

missing_columns = set(temperature_columns + humidity_columns) - set(df_clean.columns)
if missing_columns:
    raise KeyError(f'В датасете нет столбцов: {sorted(missing_columns)}')

In [6]:
df_clean['indoor_temperature_mean'] = df_clean[temperature_columns].mean(axis=1)
df_clean['indoor_temperature_min'] = df_clean[temperature_columns].min(axis=1)
df_clean['indoor_temperature_max'] = df_clean[temperature_columns].max(axis=1)
df_clean['indoor_temperature_range'] = (
    df_clean['indoor_temperature_max'] - df_clean['indoor_temperature_min']
)

df_clean['indoor_humidity_mean'] = df_clean[humidity_columns].mean(axis=1)
df_clean['indoor_humidity_min'] = df_clean[humidity_columns].min(axis=1)
df_clean['indoor_humidity_max'] = df_clean[humidity_columns].max(axis=1)
df_clean['indoor_humidity_range'] = (
    df_clean['indoor_humidity_max'] - df_clean['indoor_humidity_min']
)

df_clean['indoor_outdoor_temperature_diff'] = df_clean['indoor_temperature_mean'] - df_clean['T_out']
df_clean['indoor_outdoor_humidity_diff'] = df_clean['indoor_humidity_mean'] - df_clean['RH_out']

aggregate_features = [
    'indoor_temperature_mean',
    'indoor_temperature_min',
    'indoor_temperature_max',
    'indoor_temperature_range',
    'indoor_humidity_mean',
    'indoor_humidity_min',
    'indoor_humidity_max',
    'indoor_humidity_range',
    'indoor_outdoor_temperature_diff',
    'indoor_outdoor_humidity_diff',
]

df_clean[aggregate_features].head()

,indoor_temperature_mean,indoor_temperature_min,indoor_temperature_max,indoor_temperature_range,indoor_humidity_mean,indoor_humidity_min,indoor_humidity_max,indoor_humidity_range,indoor_outdoor_temperature_diff,indoor_outdoor_humidity_diff
0,18.435000,17.033333,19.89,2.856667,46.742500,41.626667,55.20,13.573333,11.835000,-45.257500
1,18.439167,17.066667,19.89,2.823333,46.672708,41.560000,55.20,13.640000,11.955833,-45.327292
2,18.421667,17.000000,19.89,2.890000,46.562917,41.433333,55.09,13.656667,12.055000,-45.437083
3,18.396250,17.000000,19.89,2.890000,46.468750,41.290000,55.09,13.800000,12.146250,-45.531250
4,18.408750,17.000000,19.89,2.890000,46.462917,41.230000,55.09,13.860000,12.275417,-45.537083


In [7]:
assert df_clean[aggregate_features].isna().sum().sum() == 0, 'В новых признаках есть пропуски'
assert (df_clean['indoor_temperature_range'] >= 0).all(), 'Некорректный разброс температуры'
assert (df_clean['indoor_humidity_range'] >= 0).all(), 'Некорректный разброс влажности'

df_clean[aggregate_features].describe().T

,count,mean,std,min,25%,50%,75%,max
indoor_temperature_mean,19735.0,20.815611,1.812567,16.012708,19.663000,20.597500,21.764375,26.061940
indoor_temperature_min,19735.0,18.993190,1.866267,14.890000,17.760000,18.600000,20.100000,24.500000
indoor_temperature_max,19735.0,22.779580,1.965423,17.290000,21.587302,22.600000,23.963333,29.856667
indoor_temperature_range,19735.0,3.786390,0.907657,1.260000,3.132048,3.700000,4.320278,7.770000
indoor_humidity_mean,19735.0,41.221952,3.977196,30.842798,38.354583,40.572708,44.162176,53.947153
indoor_humidity_min,19735.0,35.067213,4.863485,20.463333,31.390000,34.590000,38.565833,47.218000
indoor_humidity_max,19735.0,51.143511,8.920412,35.363333,45.560000,49.320000,53.900000,96.321667
indoor_humidity_range,19735.0,16.076298,8.559847,4.984286,11.000000,13.718000,17.642778,63.821667
indoor_outdoor_temperature_diff,19735.0,13.403946,4.202446,-1.280083,10.668958,13.589667,16.293631,24.172708
indoor_outdoor_humidity_diff,19735.0,-38.528466,13.662339,-61.735631,-49.001875,-41.474653,-29.673345,9.981333


### Лаги, защита от утечек

In [8]:
# Лаги потребления (шаг 10 минут: 1 = 10 мин, 2 = 20 мин, 3 = 30 мин, 144 = сутки)
df_clean['Appliances_lag_1'] = df_clean['Appliances'].shift(1)
df_clean['Appliances_lag_2'] = df_clean['Appliances'].shift(2)
df_clean['Appliances_lag_3'] = df_clean['Appliances'].shift(3)
df_clean['Appliances_lag_144'] = df_clean['Appliances'].shift(144)

# Лаг освещения (используем прошлое значение вместо текущего)
df_clean['lights_lag_1'] = df_clean['lights'].shift(1)

# Удаляем исходный столбец 'lights', но оставляем 'date' до разбиения по времени
df_clean = df_clean.drop(columns=['lights'])

# Убираем начальные строки с NaN после сдвигов и сбрасываем индекс
df_clean = df_clean.dropna().reset_index(drop=True)

print(f"Датафрейм готов. Строк: {len(df_clean)}, Колонок: {len(df_clean.columns)}")

Датафрейм готов. Строк: 19591, Колонок: 52


In [9]:
lag_cols = [
    "Appliances_lag_1",
    "Appliances_lag_2",
    "Appliances_lag_3",
    "Appliances_lag_144",
    "lights_lag_1",
]
lag_descriptions = {
    "Appliances_lag_1": "Потребление 10 мин назад (1 шаг)",
    "Appliances_lag_2": "Потребление 20 мин назад (2 шага)",
    "Appliances_lag_3": "Потребление 30 мин назад (3 шага)",
    "Appliances_lag_144": "Потребление 24 ч назад (144 шага)",
    "lights_lag_1": "Освещение 10 мин назад (защита от утечки)",
}

lag_summary = pd.DataFrame({
    "Описание": pd.Series(lag_descriptions),
    "dtype": df_clean[lag_cols].dtypes.astype(str),
    "min": df_clean[lag_cols].min().round(1),
    "median": df_clean[lag_cols].median().round(1),
    "max": df_clean[lag_cols].max().round(1),
})

print(f"Датафрейм готов. Строк: {len(df_clean)}, Колонок: {len(df_clean.columns)}")
display(lag_summary)

Датафрейм готов. Строк: 19591, Колонок: 52


,Описание,dtype,min,median,max
Appliances_lag_1,Потребление 10 мин назад (1 шаг),float64,10.0,60.0,1080.0
Appliances_lag_2,Потребление 20 мин назад (2 шага),float64,10.0,60.0,1080.0
Appliances_lag_3,Потребление 30 мин назад (3 шага),float64,10.0,60.0,1080.0
Appliances_lag_144,Потребление 24 ч назад (144 шага),float64,10.0,60.0,1080.0
lights_lag_1,Освещение 10 мин назад (защита от утечки),float64,0.0,0.0,50.0


### Разделение на Train и Test

Поскольку данные представляют собой временной ряд с шагом 10 минут, случайное перемешивание (`train_test_split` со `shuffle=True`) недопустимо — это привело бы к утечке данных из будущего в прошлое и искусственному завышению качества моделей.

* **Принцип разбиения:** данные отсортированы строго по возрастанию времени (`date`), первые 80% наблюдений отнесены к обучающей выборке (`train`), последние 20% — к тестовой (`test`).
* **Учёт неполных месяцев:** исходный период охватывает неполные календарные месяцы (с середины января по конец мая 2016 года), поэтому деление выполняется по хронологическому порядку записей, а признак «месяц» не используется.
* **Результат:** обучающая выборка покрывает период с 12 января по 30 апреля 2016 года (15 672 записи), тестовая — с 30 апреля по 27 мая 2016 года (3 919 записей) без пересечений по времени.

In [10]:
# Проверяем, что данные отсортированы по времени
df_clean = df_clean.sort_values('date').reset_index(drop=True)

# Граница разбиения: первые 80% по времени — train, последние 20% — test
split_idx = int(len(df_clean) * 0.8)

train_df = df_clean.iloc[:split_idx].copy()
test_df = df_clean.iloc[split_idx:].copy()

print(f"Train: {len(train_df)} строк | с {train_df['date'].min()} по {train_df['date'].max()}")
print(f"Test:  {len(test_df)} строк | с {test_df['date'].min()} по {test_df['date'].max()}")

Train: 15672 строк | с 2016-01-12 17:00:00 по 2016-04-30 12:50:00
Test:  3919 строк | с 2016-04-30 13:00:00 по 2016-05-27 18:00:00


In [11]:
target_col = 'Appliances'
drop_from_X = ['Appliances', 'date']

# Матрицы признаков и целевые переменные
X_train = train_df.drop(columns=drop_from_X).copy()
y_train = train_df[target_col].copy()

X_test = test_df.drop(columns=drop_from_X).copy()
y_test = test_df[target_col].copy()

# Логарифмированная цель для моделей, которым важна нормализация распределения
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Проверка размеров и типов данных
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print("Нечисловых столбцов в X_train:", X_train.select_dtypes(exclude='number').columns.tolist())

# Сохранение обработанных выборок
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

train_df.to_parquet(f"{output_dir}/train.parquet", index=False)
test_df.to_parquet(f"{output_dir}/test.parquet", index=False)

print(f"Файлы сохранены в '{output_dir}/' (train.parquet, test.parquet)")

X_train: (15672, 50), y_train: (15672,)
X_test:  (3919, 50),  y_test:  (3919,)
Нечисловых столбцов в X_train: []
Файлы сохранены в '../data/processed/' (train.parquet, test.parquet)


In [12]:
assert X_train.isna().sum().sum() == 0, "Найдены NaN в X_train"
assert X_test.isna().sum().sum() == 0, "Найдены NaN в X_test"
assert list(X_train.columns) == list(X_test.columns), "Колонки X_train и X_test не совпадают"
assert 'Appliances' not in X_train.columns and 'date' not in X_train.columns, "Лишний столбец попал в X"

print("Проверка пройдена успешно:")
print(f"- Признаков в X: {X_train.shape[1]}")
print(f"- Пропусков в X_train / X_test: 0 / 0")
print(f"- Колонки train и test совпадают полностью")

Проверка пройдена успешно:
- Признаков в X: 50
- Пропусков в X_train / X_test: 0 / 0
- Колонки train и test совпадают полностью
